In [1]:
!pip install -q langchain langchain-google-genai pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.3/73.3 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 565.9/565.9 kB 12.9 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

In [3]:
api_key = userdata.get("GEMINI_API_KEY")

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    google_api_key=api_key
)

In [4]:
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a social-media writer for a small Cairo-based agency. "
        "You create polished Instagram posts for local cafés and shops."
    ),
    (
        "human",
        "Create one Instagram post draft based on this product description:\n\n"
        "{description}\n\n"
        "Include a punchy title, an engaging caption under 280 characters, "
        "3 to 5 relevant hashtags, and short alt-text for accessibility."
    )
])

In [5]:
chain = prompt | llm

test_description = "New iced karak chai with cardamom, served cold in a tall glass."

response = chain.invoke({
    "description": test_description
})

print(response.content)

[{'type': 'text', 'text': '**Title:**  \nKarak, but make it iced. 🧊✨\n\n**Caption:**  \nThe Cairo heat is no match for this. ☀️ Our new Iced Karak Chai is freshly brewed with aromatic cardamom, poured over ice, and served tall. Spicy, creamy, and seriously refreshing—your new afternoon ritual starts now. \n\n**Hashtags:**  \n#CairoCafes #IcedKarak #CairoFoodies #KarakChai #NewInCairo\n\n**Alt-Text:**  \nA tall, condensation-beaded glass filled with creamy, light-brown iced karak chai, served over ice on a rustic wooden café table.', 'extras': {'signature': 'EpcXCpQXARFNMg8buMI6pRLaG3UF4UogW3cjkJwe5HuAQl+FrIZTERvZjioQ7vRDTUwNiosgNKzD0lHtCwsxJI7Scv9TkiUsdUq/QxtqcK1rCtuppI16loa984P7U2Dc0Es0OBb7jwajRRDDkrKR57bQu7MDdynCUFBX+ckgOzwTVQdiiKmTwhJV5HTbuzVIYj7/FpZQTx/nZAYO9uZRBUw9gwqOJevYSqwuwyI3jiN+B+DOt8ZDBIHBx+rWVYJFoH1h8QagvnL/JwSdfAsVjBFsCWhQDxj7ALWaTJld9bxrABgtB07R7SOOnoAUbbo3QstA+T0c0s0sB/suVIJvSr69ydB3UselWzWbw6shi0xDb4Y6Ax570aeFfy+bp+7Sr1oAy7TXv5UjxFbmS7x3pStdzxCrlfsMlPgkH6AtSjLNQWyU48Fk

In [6]:
class SocialPost(BaseModel):
    title: str = Field(
        description="A punchy social media title, maximum 60 characters."
    )
    caption: str = Field(
        description="An engaging social media caption, maximum 280 characters."
    )
    hashtags: list[str] = Field(
        description="3 to 5 relevant hashtags, each starting with #."
    )
    alt_text: str = Field(
        description="A short accessibility description of the post image."
    )

In [7]:
structured_llm = llm.with_structured_output(SocialPost)
structured_chain = prompt | structured_llm

In [8]:
result = structured_chain.invoke({
    "description": test_description
})

print(result)
print(type(result))

title='Chilled to Perfection: Our New Iced Karak' caption="Cairo's summer just got a whole lot cooler! Try our brand-new Iced Karak Chai. Brewed with rich spices and freshly ground cardamom, served ice-cold in a tall glass. It is the refreshing kick you need today. Drop by and grab yours!" hashtags=['#IcedKarak', '#CairoCafes', '#KarakChai', '#CairoFoodies'] alt_text='A tall glass of creamy iced karak chai with ice cubes and a sprinkle of cardamom on top, set against a cozy café background.'
<class '__main__.SocialPost'>


In [9]:
print("Title:", result.title)
print("Caption:", result.caption)
print("Hashtags:", result.hashtags)
print("Alt-text:", result.alt_text)

Title: Chilled to Perfection: Our New Iced Karak
Caption: Cairo's summer just got a whole lot cooler! Try our brand-new Iced Karak Chai. Brewed with rich spices and freshly ground cardamom, served ice-cold in a tall glass. It is the refreshing kick you need today. Drop by and grab yours!
Hashtags: ['#IcedKarak', '#CairoCafes', '#KarakChai', '#CairoFoodies']
Alt-text: A tall glass of creamy iced karak chai with ice cubes and a sprinkle of cardamom on top, set against a cozy café background.


In [10]:
test_descriptions = [
    "New iced karak chai with cardamom, served cold in a tall glass.",
    "Creamy iced karak chai made with black tea, milk, and fragrant cardamom.",
    "Refreshing iced karak chai served over ice with a sprinkle of ground cardamom."
]

print("Test descriptions prepared:")
for description in test_descriptions:
    print("-", description)

Test descriptions prepared:
- New iced karak chai with cardamom, served cold in a tall glass.
- Creamy iced karak chai made with black tea, milk, and fragrant cardamom.
- Refreshing iced karak chai served over ice with a sprinkle of ground cardamom.


In [11]:
translate_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a professional Arabic translator for a Cairo-based social media agency."
    ),
    (
        "human",
        "Translate the following social media caption into natural, engaging Arabic. "
        "Preserve the original meaning and promotional tone. "
        "Return only the Arabic translation, with no explanations.\n\n"
        "{caption}"
    )
])

In [12]:
translation_chain = translate_prompt | llm

In [13]:
karak_result = structured_chain.invoke({
    "description": test_descriptions[0]
})

arabic_response = translation_chain.invoke({
    "caption": karak_result.caption
})

print(arabic_response.content)

[{'type': 'text', 'text': 'اكسر حر القاهرة مع شاي الكرك المثلج الجديد بتاعنا! 🧊 \n\nمتولّف بتوابل غنية ولمسة حبهان ريحتها تجنن، هو ده الانتعاش الكريمي اللي محتاجه يفصلك عن حر اليوم. \n\nعدّي علينا دلوقتي وجرّب كوبايتك المثلجة! ☕✨', 'extras': {'signature': 'EookCockARFNMg811RhA5IMv6Z0v0r+f1O7EVQ9GRC5SEwtvYeJVT8MYopceAx1nj/8IH1BdkXrU1DThTtpobyPOkDdDtIhYTxCT46Oz4k/8sooA1cm7+hbB3nY7EIgBtWW1cSZNmz7vPt1ryTOUPi++2C/Bxw/EB6e9T/Uc+vGRea3rDhH901hJMq7T8w6STG9mEIxyTq5krYtQJz2GoGkvNGMwSdgfXJwIt6Mf0OYyW3zfZhRG2paOJqqaCxXWOjUQk6sgJKZxJsQSWCDa1Dm+Q+B8bsH9CEvOcki6zF/HCM6WVIEXuyglolyt4p+dLnOM5BGe17vfE5v9j5VfE/6L1rJCmdgJUJ58kli3XpCc72GgYQP43RvpQ5GQI5maaE3NFAOWBe5VvhAuRCOC4Nk15kbvqge+VyeqUVbjsAummnhCUhP1YHWLrw87UQtiGzgGEVDDxizRlKNeylE5i7yEigV8Jp03nTi96s9XZHJt07gj/TJYkB6CYxlddmKFzFWPlwBYlpymQzuc792KREIlp+/UUJ+cj27bAefxO72i2MRHjq65Mlh8Kc0h+GiCqVhNkAmG6F2Jm2cj3x8hVUYeoIrf6YE0Lk8K0T4k8zgmTfbTJPxse1ANgLm3w5U6pNn7q872WD4cSHd8hAKa3bLoMHxhcJdZDHYB+0a1zQs7IPFu2KJsXDisZ0MRe668udnAPxkIOO8L6WZN/XjTzFfF2gQuRZka5rvuI6U

In [14]:
def validate_social_post(post):
    checks = {
        "title_under_60": len(post.title) <= 60,
        "caption_under_280": len(post.caption) <= 280,
        "hashtags_3_to_5": 3 <= len(post.hashtags) <= 5,
        "hashtags_start_with_#": all(tag.startswith("#") for tag in post.hashtags),
        "alt_text_not_empty": bool(post.alt_text.strip())
    }

    return checks


validation = validate_social_post(karak_result)

for check, passed in validation.items():
    print(f"{check}: {'PASS' if passed else 'FAIL'}")

title_under_60: PASS
caption_under_280: PASS
hashtags_3_to_5: PASS
hashtags_start_with_#: PASS
alt_text_not_empty: PASS


In [15]:
all_checks_passed = all(validation.values())

print("Overall validation:", "PASS" if all_checks_passed else "FAIL")

Overall validation: PASS


In [16]:
print("TITLE:")
print(karak_result.title)

print("\nCAPTION:")
print(karak_result.caption)

print("\nHASHTAGS:")
print(" ".join(karak_result.hashtags))

print("\nALT-TEXT:")
print(karak_result.alt_text)

print("\nARABIC CAPTION:")
print(arabic_response.content)

TITLE:
Cairo Heatwave? Meet Our New Iced Karak!

CAPTION:
Beat the Cairo heat with our brand-new Iced Karak Chai! 🧊 Steeped with bold spices and a fragrant kick of cardamom, it’s the refreshing, creamy escape you need today. Swing by and grab a cold glass! ☕✨

HASHTAGS:
#IcedKarak #CairoCafes #CardamomChai #CairoEats

ALT-TEXT:
A tall glass of creamy iced karak chai with ice cubes and a dash of cardamom on top, sitting on a wooden table in a sunlit Cairo café.

ARABIC CAPTION:
[{'type': 'text', 'text': 'اكسر حر القاهرة مع شاي الكرك المثلج الجديد بتاعنا! 🧊 \n\nمتولّف بتوابل غنية ولمسة حبهان ريحتها تجنن، هو ده الانتعاش الكريمي اللي محتاجه يفصلك عن حر اليوم. \n\nعدّي علينا دلوقتي وجرّب كوبايتك المثلجة! ☕✨', 'extras': {'signature': 'EookCockARFNMg811RhA5IMv6Z0v0r+f1O7EVQ9GRC5SEwtvYeJVT8MYopceAx1nj/8IH1BdkXrU1DThTtpobyPOkDdDtIhYTxCT46Oz4k/8sooA1cm7+hbB3nY7EIgBtWW1cSZNmz7vPt1ryTOUPi++2C/Bxw/EB6e9T/Uc+vGRea3rDhH901hJMq7T8w6STG9mEIxyTq5krYtQJz2GoGkvNGMwSdgfXJwIt6Mf0OYyW3zfZhRG2paOJqqaCxXWOjUQk

In [17]:
import json

karak_output = {
    "description": test_descriptions[0],
    "title": karak_result.title,
    "caption": karak_result.caption,
    "hashtags": karak_result.hashtags,
    "alt_text": karak_result.alt_text,
    "arabic_caption": arabic_response.content
}

with open("karak_result.json", "w", encoding="utf-8") as f:
    json.dump(karak_output, f, ensure_ascii=False, indent=2)

print("Karak result saved successfully.")

Karak result saved successfully.


In [18]:
with open("karak_result.json", "r", encoding="utf-8") as f:
    saved_karak = json.load(f)

print(json.dumps(saved_karak, ensure_ascii=False, indent=2))

{
  "description": "New iced karak chai with cardamom, served cold in a tall glass.",
  "title": "Cairo Heatwave? Meet Our New Iced Karak!",
  "caption": "Beat the Cairo heat with our brand-new Iced Karak Chai! 🧊 Steeped with bold spices and a fragrant kick of cardamom, it’s the refreshing, creamy escape you need today. Swing by and grab a cold glass! ☕✨",
  "hashtags": [
    "#IcedKarak",
    "#CairoCafes",
    "#CardamomChai",
    "#CairoEats"
  ],
  "alt_text": "A tall glass of creamy iced karak chai with ice cubes and a dash of cardamom on top, sitting on a wooden table in a sunlit Cairo café.",
  "arabic_caption": [
    {
      "type": "text",
      "text": "اكسر حر القاهرة مع شاي الكرك المثلج الجديد بتاعنا! 🧊 \n\nمتولّف بتوابل غنية ولمسة حبهان ريحتها تجنن، هو ده الانتعاش الكريمي اللي محتاجه يفصلك عن حر اليوم. \n\nعدّي علينا دلوقتي وجرّب كوبايتك المثلجة! ☕✨",
      "extras": {
        "signature": "EookCockARFNMg811RhA5IMv6Z0v0r+f1O7EVQ9GRC5SEwtvYeJVT8MYopceAx1nj/8IH1BdkXrU1DThTtp

In [19]:
arabic_caption = arabic_response.content[0]["text"]

print(arabic_caption)

اكسر حر القاهرة مع شاي الكرك المثلج الجديد بتاعنا! 🧊 

متولّف بتوابل غنية ولمسة حبهان ريحتها تجنن، هو ده الانتعاش الكريمي اللي محتاجه يفصلك عن حر اليوم. 

عدّي علينا دلوقتي وجرّب كوبايتك المثلجة! ☕✨


In [20]:
def generate_post(description):
    result = structured_chain.invoke({
        "description": description
    })

    arabic_response = translation_chain.invoke({
        "caption": result.caption
    })

    arabic_caption = arabic_response.content[0]["text"]

    return {
        "description": description,
        "title": result.title,
        "caption": result.caption,
        "hashtags": result.hashtags,
        "alt_text": result.alt_text,
        "arabic_caption": arabic_caption
    }

In [21]:
test_descriptions = [
    "New iced karak chai with cardamom, served cold in a tall glass.",
    "Creamy iced karak chai made with black tea, milk, and fragrant cardamom.",
    "Refreshing iced karak chai served over ice with a sprinkle of ground cardamom."
]

posts = []

for description in test_descriptions:
    post = generate_post(description)
    posts.append(post)

    print(post)
    print("-" * 80)

{'description': 'New iced karak chai with cardamom, served cold in a tall glass.', 'title': 'Beat the Cairo Heat with Iced Karak!', 'caption': 'Your favorite spiced comfort, now served ice-cold. Introducing our new Iced Karak Chai—brewed with rich black tea, creamy milk, and freshly crushed cardamom. It is the perfect refreshing pick-me-up for sunny Cairo afternoons. Swing by and grab yours today!', 'hashtags': ['#CairoCafes', '#IcedKarak', '#KarakChai', '#CairoEats', '#SummerInCairo'], 'alt_text': 'A tall, condensation-beaded glass filled with creamy, iced karak chai tea, served over ice cubes with a sprinkle of ground cardamom on top.', 'arabic_caption': 'دفاك المفضل بنكهة التوابل الغنية، دلوقتي بنقدمهولك مثلج! 🧊\n\nبنقدملكم "شاي كرك مثلج" الجديد—محضر من الشاي الأسود الغني، الحليب الكريمي، والحبهان المطحون طازة. هو جرعة الانتعاش المثالية اللي هتعدل مزاجك في عصاري القاهرة المشمسة. ☀️\n\nعدّي علينا واطلب كوبايتك النهاردة!'}
--------------------------------------------------------------

In [22]:
import pandas as pd

df = pd.DataFrame(posts)

df["hashtags"] = df["hashtags"].apply(lambda x: ", ".join(x))

df.to_csv("posts.csv", index=False)

df

,description,title,caption,hashtags,alt_text,arabic_caption
0,"New iced karak chai with cardamom, served cold...",Beat the Cairo Heat with Iced Karak!,"Your favorite spiced comfort, now served ice-c...","#CairoCafes, #IcedKarak, #KarakChai, #CairoEat...","A tall, condensation-beaded glass filled with ...",دفاك المفضل بنكهة التوابل الغنية، دلوقتي بنقدم...
1,"Creamy iced karak chai made with black tea, mi...",Beat the Cairo Heat: Iced Karak Chai is Here! 🧊,A frosty twist on your favorite classic. Our n...,"#CairoCafes, #IcedKarak, #ZamalekEats, #CairoF...","A tall glass of creamy, beige iced karak chai ...",لمسة باردة ومنعشة لمشروبك الكلاسيكي المفضل! ❄️...
2,Refreshing iced karak chai served over ice wit...,Beat the Cairo Heat with Iced Karak Chai! ❄️,Summer in Cairo calls for a refreshing twist o...,"#CairoCafes, #IcedKarakChai, #CairoEats, #Kara...","A tall, frosted glass of creamy, golden-brown ...",صيف القاهرة محتاج لمسة انتعاش جديدة على مشروبك...


In [23]:
print("Rows:", len(df))
print("Columns:", list(df.columns))
print()
print(df)

Rows: 3
Columns: ['description', 'title', 'caption', 'hashtags', 'alt_text', 'arabic_caption']

                                         description  \
0  New iced karak chai with cardamom, served cold...   
1  Creamy iced karak chai made with black tea, mi...   
2  Refreshing iced karak chai served over ice wit...   

                                             title  \
0             Beat the Cairo Heat with Iced Karak!   
1  Beat the Cairo Heat: Iced Karak Chai is Here! 🧊   
2     Beat the Cairo Heat with Iced Karak Chai! ❄️   

                                             caption  \
0  Your favorite spiced comfort, now served ice-c...   
1  A frosty twist on your favorite classic. Our n...   
2  Summer in Cairo calls for a refreshing twist o...   

                                            hashtags  \
0  #CairoCafes, #IcedKarak, #KarakChai, #CairoEat...   
1  #CairoCafes, #IcedKarak, #ZamalekEats, #CairoF...   
2  #CairoCafes, #IcedKarakChai, #CairoEats, #Kara...   

            

In [24]:
for _, row in df.iterrows():
    assert len(row["title"]) <= 60
    assert len(row["caption"]) <= 280

    hashtags = row["hashtags"].split(", ")
    assert 3 <= len(hashtags) <= 5
    assert all(tag.startswith("#") for tag in hashtags)

    assert isinstance(row["arabic_caption"], str)
    assert len(row["arabic_caption"].strip()) > 0

print("All validation checks passed!")

All validation checks passed!
